In [21]:
import subprocess
import warnings
warnings.filterwarnings('ignore')
import re
import os
import pandas as pd
import duckdb
import shutil
import json
import os
import snowflake.connector



### **BUILD/MIGRATE NEW PLATFORM DBT OBJECTS**

1. **Generate ALL Objects Lineage for the DEs to be migrated.**
    - For Service we picked 2 DEs and generated two data frames : __final_df_models__ and __final_df_src__, these two df holds information of all the models and sources required to build a DE(basically the whole lineage master data).
    - These two df are generated by running _dbt list --select +<pub_view>_
2. **Segregate Gold, Silver, Bronze models.**
    - Once we have the lineage in the dataframes, next step is to segregate them into their respective new database. We would have 3 list holding models for each db i.e bronze,silver,gold.
        1. _BRONZE_ 
            -  Holds All ODS models with new names.
            -  New table alias format: [ods schema name]_TABLE_NAME. Example: new alias: SAPAPI_ZCDV_T_CMTU024_ACDOCA, CRM_FILE_OPPORTUNITY.
            -  dbt Model name format: [database-name]__[schema_name]\__[table_alias].
        2. _SILVER_
            -   Holds Common Models (reusable models like SAPH), 1 to 1 Common Reports, Legacy OTLs.
            -   dbt Model name format: [database-name]__[schema_name]\__[table_alias].
            -   In case of duplicate model name scenario, pls add source_name prefix in the table_alias. (Example: Company table is present in mdm and myskills, make it Mdm_company and Myskills_company).
        3. _GOLD_
            -   Holds Enterprise Reports.
            -   dbt Model name format: [database-name]__[schema_name]\__[table_alias].
3.  **Models Renaming: generating master mapping "new_name".**
    - Now we have our master data lineage in __final_df_models__ and list of bronze, silver, gold models in their respective varibales, Its time to rename those based on new format as mentioned above in 2. We would create a master mapping dictionary called __new_name__ which will hold {old name : new name} mapping which would be used for multiple upcoming steps.
    1.  Find models with duplicate alias, so we can add source prefix
    2.  Special case: A list of models for which we need to add a specific prefix, this was a speical reqest.
    3.  Load dbt _manifest.json_ as this will be used to generate the NEW_NAME MAPPING and then to generate the NEW dbt models.
    4.  Generate NEW_NAME MAPPING.
4.  **New dbt Models Generation**
    -   Here we run through the manifest.json and new_names dictionary and pick the old model sql , replace the old refs, alias, model_name  with the new ones using the mapping and finally write the SQL file into the new dbt project location.
5.  **New Source Schema: STATIC_OBJECTS:** 
    -   There are some sources like EDW_DE, PUB_COMMON_SYSTEMS etc (refer dict: *schema_mappings*) which are not refreshed and are static in snowflake. To simplify migration and better consolidation of these, we created a new source schema: *STATIC_OBJECTS* which will hold all objects which are one time loads and doest refresh(This also includes all OTL Legacy base tables).
    -   We replace the source references with new *STATIC_OBJETCS* schema in the new dbt models generated.
6.  **Build new Objects in New Databases (Clone Prod):**
    -   Using new_names mapping dict we generate clone scripts for our new objects and create those in snowflake, basically we clone from prod.
7.  **Create OLD V/s NEW MAPPING Document**
    -   As we created new objects with new names in new locations, it would be indispensable to provide users with a mapping document that outlines the old and new name association. Using *new_names* dictionary and a *result_dict* variable which we created when we segregated gold models - we generate a mapping excel. (PLEASE ADD LEGACY OTLS VIEWS IN THIS DOCUMENT MANUALLY)
8.  **Parrallel Testing**
    -   using the MAPPING Document, we hit Snowflake to fetch the counts for old and new objects and generate a report (adding 3 new column in existing mapping doc- old_count,new_count,is_count_same). 
9.  **Generate DDls for LEGACY one time load VIEWS in SILVER**
    -   Use the code to generate DDLs to migrate legacy one time load views into SILVER.

----------------------------------------------WIKI ENDS------------------------------------------

Generate ALL Objects Lineage for the reports to be migrated - STARTS

In [22]:
def count_ref_occurrences_gold(folder_path,model_list):
    ref_count_dict = {}

    for root, dirs, files in os.walk(folder_path):
        for filename in files:
            file = filename.replace('.SQL','')
            if filename.endswith(".SQL") and file in model_list:
                file_path = os.path.join(root, filename)
                with open(file_path, 'r', encoding='utf-8') as file:
                    content = file.read()
                    count = content.count("{{ ref('")

                ref_count_dict[filename] = count

    return ref_count_dict

In [23]:
de_paths = ['de_ads_alarm_analysis','de_ads_etch_auto_tools','de_ads_parts_order_history']

project_dir = r'C:\Users\223163\Documents\master_repo\EnterpriseCloud\astro_datautility\dags\dbt\dbt_project'
pub_path = os.path.join(project_dir,"models\pub")

new_schema_name = "TEA_GCE_OPERATIONS"

In [24]:
#1.GET LIST OF REPORTS TO BE MIGRATED INTO NEW DBT PROJECT
pub_de = []
for de in de_paths:
    de_path = os.path.join(pub_path,de)
    for file in os.listdir(de_path):
        if "PUB__" in file:
            file = file.replace(".SQL",'')
            pub_de.append(file)
            print(file)



PUB__DE_ADS_ALARM_ANALYSIS__CRM_WORKORDER_R2
PUB__DE_ADS_ETCH_AUTO_TOOLS__GLOBAL_MACHINES
PUB__DE_ADS_PARTS_ORDER_HISTORY__CONSIGNMENT_TRANSACTION
PUB__DE_ADS_PARTS_ORDER_HISTORY__PRICE_MATRIX


In [25]:
#generate the lineage for each de model
# for view in pub_de:
#    output_file = f"{view}_Lineage.txt"
#    with open(f"C:\\Users\\221291\\Documents\\New Security Project\\tea_service\\{output_file}", 'w') as f:
#       dbt_command = f"dbt list --select +{view}+"
#       print(dbt_command)

#       result = subprocess.run(dbt_command, shell=True, cwd=de_path, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

#       op = re.sub('\x1b','',result.stdout)
#       op = re.sub('\r','',op)
#       op = op.split('\n')

#       for line in op:
#          f.write(f"{line}\n")

In [26]:
#2.LOAD DBT OBJECT
from dbt.cli.main import dbtRunner,dbtRunnerResult
from dbt.contracts.graph.manifest import Manifest

res: dbtRunnerResult = dbtRunner().invoke(["parse","--project-dir",project_dir])
manifest: Manifest = res.result

dbt = dbtRunner(manifest=manifest)

07:21:59  Running with dbt=1.7.1


07:21:59  Registered adapter: snowflake=1.7.0
07:22:03  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 12 unused configuration paths:
- models.dbt_project.common.de_tee_spares_support
- models.dbt_project.pub.de_intel_sms_specialist
- models.dbt_project.pub.de_service_field
- models.dbt_project.pub.de_procurement_department
- models.dbt_project.pub.de_manufacturing
- models.dbt_project.ods.sales_operations
- models.dbt_project.pub.de_ipro
- models.dbt_project.pub.de_service
- models.dbt_project.pub.de_sales_operations
- models.dbt_project.pub.de_spares_revenue
- models.dbt_project.pub.de_intel_sms
- models.dbt_project.pub.ods_sales_operations
07:22:03  Performance info: C:\Users\223163\Documents\master_repo\EnterpriseCloud\astro_datautility\dags\dbt\dbt_project\target\perf_info.json


In [27]:
#3.RUN dbt list and generate the lineage of all the reports
final_df_models = pd.DataFrame()
final_df_src = pd.DataFrame()

for view in pub_de:
    cli_args = ["list","--select",f"+{view}","--project-dir",project_dir]
    res = dbt.invoke(cli_args)
    list_res = []
    for r in res.result:
        list_res.append(r)
    #sources list
    de_src = [src for src in list_res if "source:" in src]

    #models list
    de_models = []
    for model in list_res:
        if "source:" in model:
            break
        de_models.append(model)

    #test
    de_test = []
    for test in reversed(list_res):
        if "source:" in test:
            break
        de_test.append(test)

    de_src = [i.split(".") for i in de_src]
    de_test = [i.split(".") for i in de_test]
    de_models = [i.split(".") for i in de_models]

    df_models = pd.DataFrame(de_models,columns=['dbt project','database','schema','model'])
    df_src = pd.DataFrame(de_src,columns=['dbt project','schema','table'])
    #df_test = pd.DataFrame(de_test,columns=['dbt project','database','schema','test'])

    df_models.drop('dbt project', axis=1, inplace=True)
    df_src.drop('dbt project', axis=1, inplace=True)

    final_df_models = pd.concat([final_df_models, df_models], ignore_index=True)
    final_df_src = pd.concat([final_df_src, df_src], ignore_index=True)


07:22:04  Running with dbt=1.7.1
07:22:04  Registered adapter: snowflake=1.7.0
07:22:05  Found 4112 models, 1 operation, 238 tests, 1825 sources, 0 exposures, 0 metrics, 945 macros, 0 groups, 0 semantic models
dbt_project.common.crm.COMMON__CRM__WORKORDER_R2
dbt_project.ods.crm_file.ODS__CRM_FILE__ACCOUNT
dbt_project.ods.crm_file.ODS__CRM_FILE__BUSINESSUNIT
dbt_project.ods.crm_file.ODS__CRM_FILE__COM_TROUBLESHOOTINGDETAILS
dbt_project.ods.crm_file.ODS__CRM_FILE__GLOBALOPTIONSETMETADATA
dbt_project.ods.crm_file.ODS__CRM_FILE__INCIDENT
dbt_project.ods.crm_file.ODS__CRM_FILE__MSDYN_CUSTOMERASSET
dbt_project.ods.crm_file.ODS__CRM_FILE__MSDYN_INCIDENTTYPE
dbt_project.ods.crm_file.ODS__CRM_FILE__MSDYN_PRIORITY
dbt_project.ods.crm_file.ODS__CRM_FILE__MSDYN_PROJECT
dbt_project.ods.crm_file.ODS__CRM_FILE__MSDYN_WORKORDER
dbt_project.ods.crm_file.ODS__CRM_FILE__MSDYN_WORKORDERSUBSTATUS
dbt_project.ods.crm_file.ODS__CRM_FILE__MSDYN_WORKORDERTYPE
dbt_project.ods.crm_file.ODS__CRM_FILE__RESOURCE
db

In [28]:
#4.Save it for future use as dbt list takes time (made it persistent)
final_df_models.to_csv("final_df_models.csv",index=False)
final_df_src.to_csv("final_df_src.csv",index=False)

In [29]:
# Read it whenever required
# final_df_models = pd.read_csv('final_df_models.csv')
# final_df_src = pd.read_csv('final_df_src.csv')


Generate ALL Objects Lineage for the reports to be migrated - ENDS

Segregate Gold, Silver, Bronze models STARTS

In [30]:
#1.GOLD - get pub view 1-1 common, these common models go into gold

# folder_path = os.path.join(project_dir,"models\pub\de_service")
result_dict = {}
for de in de_paths:
    folder_path = os.path.join(pub_path,de)

    for file_name in os.listdir(folder_path):
        if file_name.endswith('.SQL'):
            file_path = os.path.join(folder_path, file_name)

            with open(file_path, 'r') as file:
                content = file.read()
                match = re.search(r"FROM\s+\{\{\s*ref\('(.+?)'\)\s*\}\}", content,re.IGNORECASE)
                if match:
                    table_name = match.group(1)
                    result_dict[file_name] = table_name
    #print(result_dict.values)

gold_models = [val for val in result_dict.values()]

gold_models_str = str(tuple(gold_models)).replace('"', "'")

In [31]:
#2.Filter GOLD :  REMOVE COMMON SAPH,KRONOS,CRM and 1:1 models from GOLD layer

folder_path = os.path.join(project_dir,"models\common")
occurrences_dict = count_ref_occurrences_gold(folder_path,gold_models)
print(f'No. of Gold Models PRE filtering: {len(gold_models)}')
for k,v in occurrences_dict.items():
    if (v == 1 or re.search(r'__CRM__|__SAPH__|__KRONOS__', k)) and not re.search(r'__DE_WISDOM__|_SNAPSHOT', k)  :
        k = k.replace('.SQL','')
        print(k)
        gold_models.remove(k)
print(f'No. of Gold Models POST filtering: {len(gold_models)}')

gold_models_str = str(tuple(gold_models)).replace('"', "'")

No. of Gold Models PRE filtering: 4
COMMON__CRM__WORKORDER_R2
COMMON__ROSETTA__CONSIGNMENT_TRANSACTION
No. of Gold Models POST filtering: 2


In [32]:
#3.SILVER - REMOVE COMMON-GOLDS FROM ALL THE COMMON, LEFT IS SILVER
result_common = duckdb.query("""select distinct database,schema,model from final_df_models where database in ('common')""").df()
if gold_models_str == '()':
    silver_df  = duckdb.query(f"""select model,schema from result_common""").df()
else:
    silver_df  = duckdb.query(f"""select model,schema from result_common where model not in {gold_models_str}""").df()
silver_models = silver_df['model'].to_list()
silver_schema = silver_df['schema'].unique()

#4.BRONZE - THIS IS JUST ODS
bronze_df  = duckdb.query(f"""select distinct database,schema,model from final_df_models where database in ('ods')""").df()
bronze_models = duckdb.query("""select distinct model from final_df_models where database in ('ods')""").df()
bronze_models = bronze_models['model'].to_list()
bronze_schema = bronze_df['schema'].unique()

print(f'len(gold_models):{len(gold_models)}\nlen(silver_models):{len(silver_models)}\nlen(bronze_models):{len(bronze_models)}')

len(gold_models):2
len(silver_models):21
len(bronze_models):89


Segregate Gold, Silver, Bronze models ENDS

Models Renaming STARTS

In [33]:
#1.FIND DUPLICATES NAMES IN COMMON
#duplicate_silver will be used soon wo add prefix and avoid duplicate names

result_common = duckdb.query("""select distinct database,schema,model from final_df_models where database in ('common')""").df()
# Trim 'COMMON__<schema>__' from the 'model' column
result_common['trimmed_model'] = result_common['model'].apply(lambda x: x.split('__', 2)[-1])

duplicates = result_common[result_common.duplicated('trimmed_model', keep=False)]
duplicate_silver = duplicates['model'].to_list()
duplicate_silver = [i for i in duplicate_silver if "__REPORTS__" not in i ] #exclude report schema duplicated name


In [34]:
#2. ADD THE MODEL NAMES IN CASE IT REQUIRES A PREFIX (SPECIAL REQUEST CASES)
# rename_stuff = ['COMMON__DE_FST_DEPARTMENTAL__CONFIGURATION',
#                 'COMMON__DE_FST_DEPARTMENTAL__XREFERENCE_CONTRACT_BUNDLETERMS',
#                 'COMMON__DE_FST_DEPARTMENTAL__XREFERENCE_CONTRACT_TYPE']

rename_stuff= []

In [35]:
#3. LOAD MANIFEST AS THIS WILL BE USED TO GENERATE NEW NAME MAPPING AND THEN THE NEW MODELS AS WELL
with open(os.path.join(project_dir,"target\\manifest.json"), 'r') as f:
    manifest = json.load(f)

In [36]:
#4. GENERATE NEW NAMES MAPPING FOR BRONZE, SILVER AND GOLD
new_names = {}
new_schema = new_schema_name
legacy_prefix = "LEGACY_FSTR"

for node in manifest['nodes'].values():
    db = node['database'].upper()
    schema = node['schema'].upper()
    table = node['alias'].upper()
    if node['name'] in bronze_models:
        new_name = '__'.join(['BRONZE',new_schema, f'{schema}_{table}'])
        new_names[node['name']] = new_name
    elif node['name'] in silver_models and node['name'] in duplicate_silver: #Duplicate handling
        new_name = '__'.join(['SILVER',new_schema, f'{schema}_{table}'])
        new_names[node['name']] = new_name
    elif node['name'] in silver_models and node['name'] in rename_stuff:  #Rename Request Handling
        new_name = '__'.join(['SILVER',new_schema, f'{legacy_prefix}_{table}'])
        new_names[node['name']] = new_name
    elif node['name'] in silver_models and node['name'] not in duplicate_silver and node['name'] not in rename_stuff:
        new_name = '__'.join(['SILVER',new_schema, table])
        new_names[node['name']] = new_name
    elif node['name'] in gold_models:
        new_name = '__'.join(['GOLD',new_schema, table])
        new_names[node['name']] = new_name

Models Renaming ENDS

Model Generation in new dbt project STARTS

In [37]:
#1. SET PATH FOR NEW DBT PROJECT and new schema
new_dbt_path = r'C:\Users\223163\Documents\master_repo\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel'
new_schema = new_schema_name.lower()

In [38]:
#2.BUILD BRONZE
for node in manifest['nodes'].values():
    db = node['database'].upper()
    schema = node['schema'].upper()
    table = node['alias'].upper()
    if node['name'] in bronze_models:
        new = new_names.get(node['name'])
        new_alias = new.split('__')[-1]
        sql = node['raw_code']
        new_sql = sql.replace('\r', '')
        pattern = r"alias\s*=\s*['\"]([^'\"]+)['\"]"
        new_sql = re.sub(pattern,f"alias='{new_alias}'", new_sql)
        # rewrites existing ref calls to use new name pattern
        for ref in node['refs']:
            ref_model_name  = ref['name']
            new_ref_model_name = new_names.get(ref_model_name, ref_model_name)
            new_ref = "{{ ref('"+new_ref_model_name+"') }}"
            ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
            new_sql = re.sub(ref_regex, new_ref, new_sql)

        new_path = f'models\\{new_schema}\\bronze'
        with open(os.path.join(new_dbt_path,new_path,f'{new}.SQL'), 'w+', encoding='utf-8') as f:
            f.write(new_sql)



In [39]:
#3.BUILD SILVER

for node in manifest['nodes'].values():
    db = node['database'].upper()
    schema = node['schema'].upper()
    table = node['alias'].upper()

    if node['name'] in silver_models:

        new = new_names.get(node['name'])
        sql = node['raw_code']
        new_sql = sql.replace('\r', '')

        if node['name'] in duplicate_silver:
            new_alias = new.split('__')[-1]
            pattern = r"alias\s*=\s*['\"]([^'\"]+)['\"]"
            new_sql = re.sub(pattern,f"alias='{new_alias}'", new_sql)

        for ref in node['refs']:
            ref_model_name  = ref['name']
            new_ref_model_name = new_names.get(ref_model_name, ref_model_name)
            new_ref = "{{ ref('"+new_ref_model_name+"') }}"
            ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
            new_sql = re.sub(ref_regex, new_ref, new_sql)

        new_path = f'models\\{new_schema}\\silver'
        with open(os.path.join(new_dbt_path,new_path,f'{new}.SQL'), 'w+', encoding='utf-8') as f:
            f.write(new_sql)

In [40]:
#4.BUILD GOLD

for node in manifest['nodes'].values():
    db = node['database'].upper()
    schema = node['schema'].upper()
    table = node['alias'].upper()
    if node['name'] in gold_models:
        new = new_names.get(node['name'])
        new_alias = new.split('__')[-1]
        sql = node['raw_code']
        new_sql = sql.replace('\r', '')

        for ref in node['refs']:
            ref_model_name  = ref['name']
            new_ref_model_name = new_names.get(ref_model_name, ref_model_name)
            new_ref = "{{ ref('"+new_ref_model_name+"') }}"
            ref_regex = '''\\{\\{ *ref *\\( *['"] *'''+ref_model_name+''' *['"] *\\) *\\}\\}'''
            new_sql = re.sub(ref_regex, new_ref, new_sql)

        new_path = f'models\\{new_schema}\\gold'
        with open(os.path.join(new_dbt_path,new_path,f'{new}.SQL'), 'w+', encoding='utf-8') as f:
            f.write(new_sql)

In [41]:
#5. REPOINT STATIC SOURCES  TO NEW LOCATION
schema_mappings = {
    "EDW__DE": "STATIC_OBJECTS",
    "PUB__COMMON_SYSTEM": "STATIC_OBJECTS",
    "COMMON__DE_SAP_MASTER_DATA": "STATIC_OBJECTS",
    "ODS__KRONOS": "STATIC_OBJECTS",
    "ODS__MDM": "STATIC_OBJECTS",
    "COMMON__TEACOMMON": "STATIC_OBJECTS",
    "ADMIN__SECURITY": "STATIC_OBJECTS",
    "ODS__BAAN": "STATIC_OBJECTS"
}

directory = os.path.join(new_dbt_path, f"models\{new_schema}")

for foldername, subfolders, filenames in os.walk(directory):
    for filename in filenames:
        if filename.endswith('.SQL'):
            file_path = os.path.join(foldername, filename)

            with open(file_path, 'r') as file:
                file_data = file.read()

            for old_schema, new_schema in schema_mappings.items():
                # Match the pattern "{{ source('old_schema', '<table_name>') }}"
                pattern = rf"\{{{{\s*source\('{old_schema}'"
                new_string = f"{{{{ source('{new_schema}'"

                file_data = re.sub(pattern, new_string, file_data)

            with open(file_path, 'w') as file:
                file.write(file_data)

print("Replacement complete!")

Replacement complete!


Model Generation in new dbt project ENDS

Build new Objects in New Databases (Clone Prod): Starts

In [42]:
#CLONING: basically Build new Objects in New Databases. (Pls replace silver bronze gold to approariate environment _dev/_qa/_sit)
final_clone = []
# exclusion = ['BRONZE__TEA_SERVICE__TIRS_ACCIDENTREPORTSFORUS_NOLOCK','SILVER__TEA_SERVICE__ACCIDENTREPORTSFORUS_NOLOCK',
#              'BRONZE__TEA_SERVICE__SAPAPI_ZCDV_T_CMTU024_ACDOCA','SILVER__TEA_SERVICE__UNIVERSAL_JOURNAL_ENTRY_LINE'
#              ,'SILVER__TEA_SERVICE__UNIVERSAL_PL_ACCOUNT_TYPE','BRONZE__TEA_SERVICE__CRM_FILE_OPPORTUNITY']
exclusion = []
for old,new in new_names.items():
    if new in exclusion:
        continue
    new_db,new_schema,new_table = new.split('__')
    old_db,old_schema,old_table = old.split('__')
    clone_sql = f"""create or replace table <<{new_db}>>.{new_schema}.{new_table} clone <<{old_db}>>.{old_schema}.{old_table};"""
    final_clone.append(clone_sql)

print(final_clone)

['create or replace table <<SILVER>>.TEA_GCE_OPERATIONS.CONTRACT_SUMMARY_DETAILS clone <<COMMON>>.CRM.CONTRACT_SUMMARY_DETAILS;', 'create or replace table <<SILVER>>.TEA_GCE_OPERATIONS.CONTRACT_SUMMARY_PURCHASE_ORDER_DETAILS clone <<COMMON>>.CRM.CONTRACT_SUMMARY_PURCHASE_ORDER_DETAILS;', 'create or replace table <<SILVER>>.TEA_GCE_OPERATIONS.OPTIONSET_METADATA clone <<COMMON>>.CRM.OPTIONSET_METADATA;', 'create or replace table <<SILVER>>.TEA_GCE_OPERATIONS.WORKORDER_R2 clone <<COMMON>>.CRM.WORKORDER_R2;', 'create or replace table <<GOLD>>.TEA_GCE_OPERATIONS.GLOBAL_MACHINES clone <<COMMON>>.DE_WISDOM.GLOBAL_MACHINES;', 'create or replace table <<GOLD>>.TEA_GCE_OPERATIONS.PRICE_MATRIX clone <<COMMON>>.REPORTS.PRICE_MATRIX;', 'create or replace table <<SILVER>>.TEA_GCE_OPERATIONS.CONSIGNMENT_TRANSACTION clone <<COMMON>>.ROSETTA.CONSIGNMENT_TRANSACTION;', 'create or replace table <<SILVER>>.TEA_GCE_OPERATIONS.BUSINESS_PARTNERS_GENERAL_DATA clone <<COMMON>>.SAPH.BUSINESS_PARTNERS_GENERAL_DA

In [43]:
# PRINT BRONZE MODELS TO CREATE TABLE USING PLATFORM ROLE
for cl in final_clone:
    if '<<BRONZE>>.' in cl:
        print(cl.replace('<<','').replace('>>','_DEV'))

create or replace table BRONZE_DEV.TEA_GCE_OPERATIONS.CRM_FILE_ACCOUNT clone ODS_DEV.CRM_FILE.ACCOUNT;
create or replace table BRONZE_DEV.TEA_GCE_OPERATIONS.CRM_FILE_BUSINESSUNIT clone ODS_DEV.CRM_FILE.BUSINESSUNIT;
create or replace table BRONZE_DEV.TEA_GCE_OPERATIONS.CRM_FILE_CNT_CNT_CONTRACTSUMMARY_ACCOUNT_FAB clone ODS_DEV.CRM_FILE.CNT_CNT_CONTRACTSUMMARY_ACCOUNT_FAB;
create or replace table BRONZE_DEV.TEA_GCE_OPERATIONS.CRM_FILE_CNT_CONTRACTSCOPEMASTER clone ODS_DEV.CRM_FILE.CNT_CONTRACTSCOPEMASTER;
create or replace table BRONZE_DEV.TEA_GCE_OPERATIONS.CRM_FILE_CNT_CONTRACTSUMMARY clone ODS_DEV.CRM_FILE.CNT_CONTRACTSUMMARY;
create or replace table BRONZE_DEV.TEA_GCE_OPERATIONS.CRM_FILE_CNT_MSDYN_AGREEMENT_MSDYN_CUSTOMERASSET clone ODS_DEV.CRM_FILE.CNT_MSDYN_AGREEMENT_MSDYN_CUSTOMERASSET;
create or replace table BRONZE_DEV.TEA_GCE_OPERATIONS.CRM_FILE_CNT_ORDERNUMBER clone ODS_DEV.CRM_FILE.CNT_ORDERNUMBER;
create or replace table BRONZE_DEV.TEA_GCE_OPERATIONS.CRM_FILE_CNT_ORDERNUMBE

In [44]:
# PRINT NON-BRONZE MODELS TO CREATE TABLE USING SCHEMA ROLE
for cl in final_clone:
    if not '<<BRONZE>>.' in cl:
        print(cl.replace('<<','').replace('>>','_DEV'))

create or replace table SILVER_DEV.TEA_GCE_OPERATIONS.CONTRACT_SUMMARY_DETAILS clone COMMON_DEV.CRM.CONTRACT_SUMMARY_DETAILS;
create or replace table SILVER_DEV.TEA_GCE_OPERATIONS.CONTRACT_SUMMARY_PURCHASE_ORDER_DETAILS clone COMMON_DEV.CRM.CONTRACT_SUMMARY_PURCHASE_ORDER_DETAILS;
create or replace table SILVER_DEV.TEA_GCE_OPERATIONS.OPTIONSET_METADATA clone COMMON_DEV.CRM.OPTIONSET_METADATA;
create or replace table SILVER_DEV.TEA_GCE_OPERATIONS.WORKORDER_R2 clone COMMON_DEV.CRM.WORKORDER_R2;
create or replace table GOLD_DEV.TEA_GCE_OPERATIONS.GLOBAL_MACHINES clone COMMON_DEV.DE_WISDOM.GLOBAL_MACHINES;
create or replace table GOLD_DEV.TEA_GCE_OPERATIONS.PRICE_MATRIX clone COMMON_DEV.REPORTS.PRICE_MATRIX;
create or replace table SILVER_DEV.TEA_GCE_OPERATIONS.CONSIGNMENT_TRANSACTION clone COMMON_DEV.ROSETTA.CONSIGNMENT_TRANSACTION;
create or replace table SILVER_DEV.TEA_GCE_OPERATIONS.BUSINESS_PARTNERS_GENERAL_DATA clone COMMON_DEV.SAPH.BUSINESS_PARTNERS_GENERAL_DATA;
create or replace t

Build new Objects in New Databases (Clone Prod): Ends

CREATE OLD V/s NEW MAPPING DOCUMENT STARTS

In [ ]:
#1.CREATE OLD VS NEW MAPPING SHEET
mapping_dict = {}
for old,new in new_names.items():
    for pub,com in result_dict.items(): #result_dict : is pub-common mapping generated in creating GOLD model step
        pub = pub.replace(".SQL",'')
        if old == com:
            mapping_dict[pub] = new
            break
    mapping_dict[old] = new

mapping_df = []
for old,new in mapping_dict.items():
    new_db,new_schema,new_table = new.split('__')
    old_db,old_schema,old_table = old.split('__')
    old_obj =f'{old_db}.{old_schema}.{old_table}'
    new_obj =f'{new_db}.{new_schema}.{new_table}'
    mapping_df.append((old_db,old_schema,old_table,old_obj,new_db,new_schema,new_table,new_obj))


df_map = pd.DataFrame(mapping_df, columns = ["OLD DATABASE","OLD SCHEMA","OLD TABLE","OLD OBJ","NEW DATABASE","NEW SCHEMA","NEW TABLE","NEW OBJ"])
df_map.to_csv("TEA_GCE_OPERATIONS.csv")

CREATE OLD V/s NEW MAPPING DOCUMENT ENDS

PARALLEL COUNT TESTING STARTS 

In [ ]:
#parrallel testing
df = pd.read_csv(r"C:\Users\223163\Documents\master_repo\EnterpriseCloud\Notebooks\TEA_GCE_OPERATIONS.csv")
df['OLD COUNT'] = None
df['NEW COUNT'] = None
new_db_env = "DEV"

conn = snowflake.connector.connect(user= "mohammad.x.yunus@us.tel.com",
                                    account="telus-global.privatelink",
                                    client_session_keep_alive = False,
                                    warehouse= "DEVELOPER_WH",
                                    role= "SYSADMIN_DEV",
                                    threads= 1,
                                    authenticator="externalbrowser",
                                    autocommit=True)

cursor = conn.cursor()
df
for idx,row in df.iterrows():
    try:
        old_count_sql = f"""select count(*) from {row["OLD DATABASE"]}.{row["OLD SCHEMA"]}.{row["OLD TABLE"]}"""
        cursor.execute(old_count_sql)
        old_cn = cursor.fetchone()[0]
        df.at[idx, 'OLD COUNT'] = old_cn
    except Exception as e:
        print(e)

conn.close()

conn = snowflake.connector.connect(user= "mohammad.x.yunus@us.tel.com",
                                    account="telus-global.privatelink",
                                    client_session_keep_alive = False,
                                    warehouse= "PLATFORM_WH",
                                    role= "PLATFORM_TEAM_DEV",
                                    threads= 1,
                                    authenticator="externalbrowser",
                                    autocommit=True)

cursor = conn.cursor()
for idx,row in df.iterrows():
    try:
        new_count_sql = f"""select count(*) from {row["NEW DATABASE"]}{new_db_env}.{row["NEW SCHEMA"]}.{row["NEW TABLE"]}"""
        cursor.execute(new_count_sql)
        new_cn = cursor.fetchone()[0]
        df.at[idx, 'NEW COUNT'] = new_cn
    except Exception as e:
        print(e)

conn.close()

df_counts = duckdb.query("""select * , "OLD COUNT"="NEW COUNT" AS IS_COUNT_SAME from df""").df()
df_counts.to_csv("TEA_GCE_OPERATIONS_Counts_11_12_3am.csv")

Generate DDls for LEGACY one time load VIEWS in SILVER

In [ ]:
#BUILD LEGACY OTL VIEW DDL IN NEW SILVER , Run this cell only if your PUB DE schema  contains any One time load legacy view,
# (those ITL views doesnt exist in dbt but will exist as a static view in pub.de_* , check in information schema if table_name starts with LEGACY_)

## create snowflake connection
conn = snowflake.connector.connect(user= "mohammad.x.yunus@us.tel.com",
                                    account="telus-global.privatelink",
                                    client_session_keep_alive = False,
                                    warehouse= "DEVELOPER_WH",
                                    role= "SYSADMIN_DEV",
                                    threads= 1,
                                    authenticator="externalbrowser",
                                    autocommit=True)

cursor = conn.cursor()

view_base_db,view_base_schema = "ODS","BAAN" #source base db and schema are mostly ODS.BAAN but pls confirm and change if needed
pub_schema = 'DE_HELPDESK'
# pub_schema = 'DE_INTEL_SMS_SPECIALIST'
match_str = 'LEGACY_'
new_db = 'SILVER_QA'
new_schema = new_schema_name
legacy_ddls = f'USE DATABASE {new_db};\nUSE SCHEMA {new_schema};\n'

try:
    sql = f"""  select table_catalog,table_schema,table_name
                from pub.information_schema.tables
                where table_schema = '{pub_schema}'
                and table_name  like '%{match_str}%';"""
    cursor.execute(sql)
    legacy_models = cursor.fetchall()

    for db,schema,tab in legacy_models:
        get_ddl = f"""select get_ddl('view','{db}.{schema}."{tab}"');"""
        cursor.execute(get_ddl)
        ddl = cursor.fetchone()[0].upper()
        ddl = ddl.replace(f"FROM {view_base_db}.{view_base_schema}.",f"FROM <<LZ>>.STATIC_OBJECTS.")
        legacy_ddls += f'{ddl}\n'

except Exception as e:
    print(tab)
    print(e)

conn.close()

PARALLEL COUNT TESTING ENDS 

*******************************IGNORE THE CODE BELOW THIS MARKDOWN*******************************